# Third CuPy example: vector addition with CuPy arrays

This third notebook computes $c_i=a_i+b_i$ with **the same input values, vector length, and single precision** as the first two CuPy notebooks and the PyCUDA examples.

The new step is to express the addition directly as `d_c = d_a + d_b`. CuPy supplies the elementwise GPU implementation and its launch configuration. We write no CUDA source and use neither `RawKernel` nor `RawModule`. This is the counterpart of the third PyCUDA notebook using `GPUArray` arithmetic.

Run the cells in order in a Colab GPU runtime. The notebook is self-contained. Validation, selected results, and optional timings appear in cells; no result files are exported.

## 1. Prepare Google Colab

Select **Runtime → Change runtime type → GPU**, then connect. The commands run on the remote Linux machine. `nvidia-smi` checks GPU availability; its CUDA version describes driver support. `nvcc --version` reports the installed toolkit compiler.

We do not invoke `nvcc` to compile a user-written kernel here. The toolkit version is useful for selecting a CuPy package if none is installed. CuPy still requires working CUDA components to execute and, when needed, generate its GPU kernels.

In [1]:
!nvidia-smi
!nvcc --version

Sat Oct 17 09:10:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Use the existing CuPy installation when available. The next cell installs a CUDA-specific binary package **only if CuPy is absent**, selecting CUDA 12 or 13 from the installed toolkit. It does not install PyCUDA or a notebook extension. Do not install multiple CuPy packages in the same environment.

If an existing CuPy installation fails to import, use a fresh Colab GPU runtime before proceeding. For other environments, follow the [CuPy installation guide](https://docs.cupy.dev/en/stable/install.html).

In [2]:
import importlib.util
import re
import subprocess
import sys

if importlib.util.find_spec("cupy") is None:
    toolkit_info = subprocess.check_output(
        ["nvcc", "--version"], text=True
    )
    match = re.search(r"release\s+(\d+)\.", toolkit_info)
    if match is None or int(match.group(1)) not in (12, 13):
        raise RuntimeError("Use a Colab GPU runtime with CUDA 12 or 13, or follow the CuPy installation guide.")
    package = f"cupy-cuda{match.group(1)}x"
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print("Installed:", package)
else:
    print("Using the existing CuPy installation.")

Using the existing CuPy installation.


## 2. Initialize CuPy

`cp` is the conventional alias for CuPy. We use device 0 throughout this standalone notebook. Unlike the PyCUDA example, there is no `pycuda.autoinit` import. CUDA failures are reported as Python exceptions.

In [3]:
import numpy as np
import cupy as cp

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA GPU is available. Select a GPU runtime in Colab.")
cp.cuda.Device(0).use()
properties = cp.cuda.runtime.getDeviceProperties(0)
device_name = properties["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()
print("GPU:", device_name)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CuPy:", cp.__version__)
print("CUDA runtime version:", cp.cuda.runtime.runtimeGetVersion())

GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
CuPy: 14.0.1
CUDA runtime version: 13000


## 3. Choose the vector length

Use $N=100003$, as in the preceding examples. Here CuPy determines the launch configuration and handles the full array length. There is no user-defined block size, grid size, bounds check, or length argument to a custom kernel. The limit below keeps the demonstration's NumPy index generation within its chosen `int32` range; it is not a general limit on CuPy arrays.

In [4]:
N = 100003
assert 1 <= N <= np.iinfo(np.int32).max
print(f"Elements: {N}")
print("Launch configuration: managed by CuPy.")

Elements: 100003
Launch configuration: managed by CuPy.


## 4. Prepare the NumPy inputs

The deterministic inputs match the previous examples: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both arrays are contiguous `float32` arrays, matching the kernel's `float` pointers.

In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
assert h_a.dtype == h_b.dtype == np.float32
assert h_a.flags.c_contiguous and h_b.flags.c_contiguous
print(f"Bytes per vector: {h_a.nbytes}")

Bytes per vector: 400012


## 5. Create device arrays and transfer the inputs

For these NumPy inputs, `cp.asarray` creates device arrays and copies their values to the GPU. Their storage remains available for later GPU operations. Applying `cp.asarray` to an already compatible CuPy array need not copy it.

Unlike the first two examples, we do not allocate an output with `cp.empty_like`: the addition expression will create it. The printout below inspects metadata only, without retrieving array values. Transfers and subsequent work use the same current stream; keep the host inputs unchanged while their copies are pending.

In [6]:
d_a = cp.asarray(h_a)
d_b = cp.asarray(h_b)
print("Device array class:", type(d_a).__name__)
print("Input shape:", d_a.shape)
print("Input dtype:", d_a.dtype)
print("Device bytes per input:", d_a.nbytes)
print("Device:", d_a.device.id)

Device array class: ndarray
Input shape: (100003,)
Input dtype: float32
Device bytes per input: 400012
Device: 0


## 6. Add the arrays on the GPU

The `+` operator acts on two `cupy.ndarray` objects, so the elementwise addition executes on the GPU. The expression creates a new device array `d_c`, leaving both inputs unchanged. It does not retrieve the result to the CPU.

The numerical operation is the same, but **the custom `vector_add` source is no longer used**. CuPy supplies the implementation and selects the launch configuration. This parallels `GPUArray` arithmetic in PyCUDA. The first use may involve compilation and other initialization. Execution can still be pending when the Python statement returns; the blocking retrieval below waits for the required work.

In [7]:
d_c = d_a + d_b
assert isinstance(d_c, cp.ndarray)
assert d_c.shape == (N,) and d_c.dtype == np.float32
print("Output shape:", d_c.shape, "dtype:", d_c.dtype)

Output shape: (100003,) dtype: float32


## 7. Retrieve and validate every output

`cp.asnumpy` explicitly retrieves the GPU result as a NumPy array. Its default blocking behavior makes that result available to the CPU before returning. We use the same independent NumPy reference and tolerances as the preceding examples and check every element, including the last entries.

In [8]:
h_c = cp.asnumpy(d_c)
h_reference = h_a + h_b
if not np.all(np.isfinite(h_c)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f"Validation passed for all {N} elements.")
print(f"Maximum absolute error: {max_abs_error:.3e}")

Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 8. Display the computed sum

The `c[i] (GPU)` column contains the result actually retrieved from the device. The first 12 and last 3 entries are displayed alongside the NumPy reference, with the same layout as the previous examples.

In [9]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')

       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 9. Optional: measure repeated CuPy array additions

As in the third PyCUDA notebook, keep the inputs on the GPU, warm up three additions, then use a host timer around 100 evaluations of `d_c = d_a + d_b` and a final wait for completion. The initial wait excludes earlier work.

The interval includes Python overhead, output-array creation, storage allocation or reuse from CuPy's memory pool, release of replaced outputs during the loop, GPU work, and the final wait. The final output remains allocated. Input creation/transfers, warm-up compilation, result retrieval, and validation are excluded.

This is **time per array-expression evaluation**, not isolated kernel time or a complete host-to-device/add/device-to-host measurement. The first two CuPy notebooks time repeated launches into a preallocated output using GPU events, so their numbers cannot directly rank these interfaces. The third PyCUDA example uses the same host-timing scope, but the libraries' allocation policies can differ. Compare on the same GPU and interpret small differences cautiously.

In [10]:
from time import perf_counter

REPEATS = 100
assert REPEATS > 0
stream = cp.cuda.get_current_stream()
for _ in range(3):
    d_c = d_a + d_b
stream.synchronize()

start = perf_counter()
for _ in range(REPEATS):
    d_c = d_a + d_b
stream.synchronize()
elapsed_ms = (perf_counter() - start) * 1000.0
mean_ms = elapsed_ms / REPEATS

# Retrieve and check the final output outside the measured interval.
h_c = cp.asnumpy(d_c)
if not np.all(np.isfinite(h_c)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
print(f"Repeated CuPy array additions: {REPEATS}")
print(f"Total host elapsed time: {elapsed_ms:.6f} ms")
print(f"Mean time per CuPy array addition: {mean_ms:.6f} ms")
print("Includes output creation/allocation or pool reuse, Python overhead, GPU work and final wait.")
print("Excludes host-device transfers; this is not isolated kernel time.")
print("Final result validated.")

Repeated CuPy array additions: 100
Total host elapsed time: 3.176778 ms
Mean time per CuPy array addition: 0.031768 ms
Includes output creation/allocation or pool reuse, Python overhead, GPU work and final wait.
Excludes host-device transfers; this is not isolated kernel time.
Final result validated.


## 10. Release the array references

Run this cell when finished. Deleting the last references releases array ownership of the device storage. CuPy's memory pool may keep freed blocks for reuse, so GPU memory reported by monitoring tools need not decrease immediately. To repeat the computation after cleanup, start again at the input-transfer cell.

In [11]:
cp.cuda.get_current_stream().synchronize()
del d_a, d_b, d_c
print("Device array references released.")

Device array references released.


## What changed from the first two examples?

- The numerical problem, input values, precision, and validation are unchanged.
- There is no user-written CUDA source, compilation call, or explicit launch configuration.
- The array expression creates the output on the GPU; the inputs remain available there.
- Host-to-device and device-to-host transfers remain explicit and retain their cost.

This is the direct counterpart of the PyCUDA `GPUArray` example. The familiar NumPy-like notation does not mean that the sum executes on the CPU: the operands are CuPy arrays. No claim is made that a built-in operation must be faster than a custom kernel.

**Preparation status:** delivered with empty outputs. Notebook structure, Python syntax, matching input/validation/display cells, and CPU-side numerical checks were verified during preparation. CUDA execution was not available in the preparation environment. Run all cells in a Colab GPU runtime to obtain actual results and timings.

References: [Basics of CuPy](https://docs.cupy.dev/en/stable/user_guide/basic.html), [cupy.add](https://docs.cupy.dev/en/stable/reference/generated/cupy.add.html), [Memory management](https://docs.cupy.dev/en/stable/user_guide/memory.html).